In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CustomerDtProcessing").getOrCreate()
spark


In [0]:
dfr = spark.read.format("csv").option("header",  "true").load("/Workspace/Users/bhagatishaanraj43@gmail.com/customers.csv")

In [0]:
dfr.show(5)

In [0]:
dfr.printSchema()

In [0]:
from pyspark.sql.functions import *
#changing date to date-time from string, is_active to boolean from string
dfr = dfr.withColumn('registration_date', to_date(col("registration_date"), 'yyyy-MM-dd')) \
      .withColumn('is_active', col("is_active").cast("boolean"))

In [0]:
dfr = dfr.fillna({'city':'Unknown', 'state':'Unknown', 'country':'Unknown'})

In [0]:
dfr = dfr.withColumn('reg_yr', year(col('registration_date'))) \
     .withColumn('reg_month', month(col('registration_date')))

In [0]:
dfr.show(5)

In [0]:
dfr.printSchema()

In [0]:
dfr.show(5)

In [0]:
unique_cities = dfr.select(countDistinct('city')).collect()
print(f'Num of unique cities:{unique_cities[0][0]}')
unique_states = dfr.select(countDistinct('state')).collect()
print(f'Num of unique states:{unique_states[0][0]}')
unique_countries = dfr.select(countDistinct('country')).collect()
print(f'Num of unique countries:{unique_countries[0][0]}')

In [0]:
dfr.groupBy('city').count().orderBy(col('count').desc()).show()

In [0]:
dfr.groupBy('state').count().orderBy(col('count').desc()).show()

In [0]:
#active inactive users count
dfr.groupBy('state').pivot('is_active').count().show()

####Applying window function

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy('state').orderBy(col('registration_date').desc())

dfr = dfr.withColumn('rank', rank().over(window_spec))\
    .withColumn('dense_rank', dense_rank().over(window_spec))\
        .withColumn('row_number', row_number().over(window_spec))


In [0]:
dfr.select('name', 'city', 'state', 'rank', 'dense_rank', 'row_number').show(5)

####getting latest data of recent customers

In [0]:
dfr_recent_cust = dfr.filter(col("registration_date") >= lit('2023-07-01'))
dfr_recent_cust.show()

In [0]:
dfr.count()

In [0]:
dfr_recent_cust.count()

####oldest and the newest customer per city

In [0]:
dfr.groupBy('city').agg(min('registration_date').alias('oldest'), max('registration_date').alias('newest')).show()

In [0]:
out_path = '/Volumes/workspace/default/processed_data_vol/customer_out.parquet'
dfr.write.mode('overwrite').parquet(out_path)

###Joining and Analyzing Customers and Orders

In [0]:
ordr_dfr = spark.read.format('csv').option("header", "true").option('inferSchema', True).load("/Workspace/Users/bhagatishaanraj43@gmail.com/orders.csv")

In [0]:
ordr_dfr.show(5)

In [0]:
ordr_dfr.printSchema()

In [0]:
customer_orders_dfr = dfr.join(ordr_dfr, 'customer_id', "inner")

In [0]:
customer_orders_dfr.count()

In [0]:
customer_orders_dfr.show(5)

In [0]:
customer_orders_dfr.display(5)

In [0]:
#Total Orders per customer

customers_ordr_count = customer_orders_dfr.groupBy('customer_id').count().orderBy(col('count').desc())
customers_ordr_count.show(10)

In [0]:
#total money spend by per consumer

custmr_total_spend = customer_orders_dfr.groupBy('customer_id').agg(sum('total_amount')).orderBy(col('sum(total_amount)').desc())
custmr_total_spend.show(10)

#####from above two cells, customer id 3336, 3884 are concluded to be the premium customers.

In [0]:
#Average spend per customer

custmr_avg_spend = customer_orders_dfr.groupBy('customer_id').agg(avg('total_amount')).orderBy(col('avg(total_amount)').desc())
custmr_avg_spend.show(10)

In [0]:
#order by status

ordr_status_count = customer_orders_dfr.groupBy('status').count()
ordr_status_count.show()

In [0]:
#Orders by month

ordr_by_month = customer_orders_dfr.withColumn('order_month', month(col('order_date')))\
    .groupBy('order_month').count()\
        .orderBy('order_month')

ordr_by_month.show()

In [0]:
# Rank the customer by total spend usingn window func

window_spc = Window.orderBy(col('sum(total_amount)').desc())

rankd_custmr = custmr_total_spend.withColumn('dense_rank', dense_rank().over(window_spc))
rankd_custmr.show(10)


In [0]:
custmr_total_spend.printSchema(), customers_ordr_count.printSchema()

In [0]:
#Finding customers with High order frequency but low total spend

custmr_spend_vs_ordr = customers_ordr_count.join(custmr_total_spend, 'customer_id','inner')\
    .orderBy(col('count').desc(), col('sum(total_amount)'))

custmr_spend_vs_ordr.show(5)

In [0]:
out_pth = '/Volumes/workspace/default/processed_data_vol/customer_and_orders_out.parquet'
customer_orders_dfr.write.mode('overwrite').parquet(out_pth)